# 02 · Gabarito humano: RePro

O RePro (Silva et al., PROPOR 2024) tem 10.003 avaliações da Americanas/B2W anotadas por humanos com tópicos e polaridade (kappa 0,68 e 0,71). É a **régua** do projeto: todos os modelos são avaliados no teste do RePro, nunca em rótulos gerados por IA.

Como a polaridade do RePro é da avaliação inteira, só usamos avaliações em que a polaridade de cada aspecto é inequívoca (ver `src/absa/repro.py`).

In [ ]:
# Configuração: no Colab, clona o repositório, instala o pacote `absa`
# e guarda cache do LLM, modelos e predições no Drive (sobrevive à queda da sessão)
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e "."
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models /content/drive/MyDrive/pln-absa-olist/interim
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/interim data/interim

In [ ]:
import pandas as pd
from absa.repro import convert, download_repro, label_distribution, load_repro

bruto = pd.read_csv(download_repro())
tudo = convert(bruto)
treino, val, teste, descartados = load_repro()
print(f"total {len(tudo)} | usadas {len(tudo) - len(descartados)} | descartadas {len(descartados)}")
print(f"treino {len(treino)} | validação {len(val)} | teste {len(teste)}")
descartados.motivo_descarte.value_counts()

## Distribuição dos rótulos no teste

In [ ]:
label_distribution(teste)

In [ ]:
# Quantos aspectos por avaliação
teste.filter(like="asp_").ne("ausente").sum(axis=1).value_counts().sort_index()

## Exemplos

In [ ]:
cols = ["texto", "asp_produto", "asp_entrega", "asp_recebimento", "asp_anuncio", "asp_outros"]
pd.set_option("display.max_colwidth", 120)
teste[cols].sample(10, random_state=3)

## Amostra opcional da Olist

`data/annotated/olist_para_anotar.csv` tem 100 comentários da Olist (estratificados por nota, 30% de pedidos atrasados) para anotar à mão seguindo `anotacao/guia_de_anotacao.md`. Serve para medir a mudança de domínio Americanas → Olist e para cobrir avaliações mistas, que o RePro descarta. Ao terminar, salve como `olist_anotado.csv`.